# Stable Diffusion

**Companion lab for MIT 15.773 Hands-On Deep Learning (Spring 2024).**

This notebook is adapted from `11b_stable_diffusion.ipynb` for study and reproducibility. The instructional cells and code are preserved, while stored outputs are removed to keep the book portable. Run cells in order and inspect shapes, metrics, and failure cases rather than treating successful execution as the only goal.

> Some labs require a GPU, external datasets, model downloads, or API credentials. Use a hosted runtime where the notebook indicates one; never commit credentials.


CREDIT: Adapted from https://github.com/fastai/diffusion-nbs/blob/master/stable_diffusion.ipynb

# Stable Diffusion with 🤗 Diffusers


This notebook shows how to use Stable Diffusion. We use the 🤗 Hugging Face [🧨 Diffusers library](https://github.com/huggingface/diffusers).


In [ ]:
!pip install -Uq diffusers

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torch
from diffusers import StableDiffusionPipeline

## Using Stable Diffusion

To run Stable Diffusion, you have to accept the model license. It's an open CreativeML OpenRail-M license that claims no rights on the outputs you generate and prohibits you from deliberately producing illegal or harmful content. The [model card](https://huggingface.co/CompVis/stable-diffusion-v1-4) provides more details. If you do accept the license, you need to be a registered user in 🤗 Hugging Face Hub and get an access token from [here](https://huggingface.co/settings/tokens) for the code to work. You can provide your access token using the `notebook_login()` function.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### Stable Diffusion Pipeline

[`StableDiffusionPipeline`](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion#diffusers.StableDiffusionPipeline) is an end-to-end [diffusion inference pipeline](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion) that allows you to start generating images with just a few lines of code.

When we say "inference" we're referring to using an existing model to generate samples (in this case, images), as opposed to "training" (or fine-tuning) models using new data.

* We use [`from_pretrained`](https://huggingface.co/docs/diffusers/main/en/api/diffusion_pipeline#diffusers.DiffusionPipeline.from_pretrained) to create the pipeline and download the pretrained weights.
* The string passed to `from_pretrained` in this case (`CompVis/stable-diffusion-v1-4`) is the repo id of a pretrained pipeline hosted on [Hugging Face Hub](https://huggingface.co/models). This is exactly what we did in the "lightning tour" of HuggingFace for standard NLP tasks in Lecture 8.
* We indicate that we want to use the `fp16` (half-precision) version of the weights, and we tell `diffusers` to expect the weights in that format. This allows us to perform much faster inference with almost no discernible difference in quality.
* The weights for all the models in the pipeline will be downloaded and cached the first time you run this cell.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4",
                                               revision="fp16",
                                               torch_dtype=torch.float16).to("cuda")

We are now ready to use the pipeline to start creating images.

In [ ]:
prompt = "a photograph of an astronaut riding a horse"

In [ ]:
torch.manual_seed(1) # set the seed for reproducibility
pipe(prompt).images[0]

In [ ]:
torch.manual_seed(1024) # if we change the seed, we will get a different image for the same prompt
pipe(prompt).images[0]

You will have noticed that running the pipeline shows a progress bar with a certain number of steps. This is the gradual denoising approach we discussed in lecture.

Give me a prompt!

In [ ]:
torch.manual_seed(1) # set the seed for reproducibility
pipe("MIT professor riding a horse").images[0]

### Negative prompts

There are many extensions and variations on the basic pipeline we saw above. We will take a quick look at one of them: *Negative prompting*. Please see the source notebook (linked to at the top) for more.

_Negative prompting_ refers to the use of two prompts and "subtracting" one prompt from another.

Let's say the first prompt is "Labrador in the style of Vermeer".

In [ ]:
torch.manual_seed(1000)
prompt = "Labrador in the style of Vermeer"
pipe(prompt).images[0]

We love the Labrador but don't care for the blue stuff.

To remove the blue stuff, we can just "subtract" the prompt "blue" from the prompt "Labrador in the style of Vermeer"(roughly speaking).

In [ ]:
torch.manual_seed(1000)
pipe(prompt, negative_prompt="blue").images[0]



---



To learn how to work with and customize diffusion models, see https://huggingface.co/docs/diffusers/en/tutorials/tutorial_overview.